# TCS Stock Price Trend Prediction - Machine Learning Benchmark
**Infosys Springboard AI-Powered Multi-Agent System for Stock Analysis**

This notebook demonstrates:
1. **Data Preprocessing & Cleaning** for NSE historical trading records (2010 - 2020).
2. **Feature Engineering** (MACD, Bollinger Bands, Volatility, Deliverable Volume %).
3. **80:20 Train-Test Split** using scikit-learn.
4. **Benchmarking 7 Machine Learning Models** (Random Forest, Gradient Boosting, XGBoost, SVM, Extra Trees, Logistic Regression, KNN).
5. **Leaderboard Evaluation & Champion Model Selection**.

In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

print("All libraries imported successfully!")

## 1. Load TCS Dataset

In [ ]:
# Load Stock_TCS.xlsx
if os.path.exists('Stock_TCS.xlsx'):
    df = pd.read_excel('Stock_TCS.xlsx')
else:
    zip_path = os.path.expanduser(r'~\Downloads\ai_powered_stock_research_dataset.zip')
    with zipfile.ZipFile(zip_path) as z:
        with z.open('Stock_TCS.xlsx') as f:
            df = pd.read_excel(f)

print(f"Raw Dataset Shape: {df.shape}")
df.head()

## 2. Feature Engineering & Preprocessing

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

close = df['Close']
df['Return_1d'] = close.pct_change()
df['Return_5d'] = close.pct_change(5)
df['SMA_5'] = close.rolling(5).mean()
df['SMA_20'] = close.rolling(20).mean()

# MACD (Moving Average Convergence Divergence)
df['EMA_12'] = close.ewm(span=12, adjust=False).mean()
df['EMA_26'] = close.ewm(span=26, adjust=False).mean()
df['MACD'] = df['EMA_12'] - df['EMA_26']
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']

# Bollinger Bands
std20 = close.rolling(20).std()
df['BB_Upper'] = df['SMA_20'] + (std20 * 2)
df['BB_Lower'] = df['SMA_20'] - (std20 * 2)
df['BB_Pct'] = (close - df['BB_Lower']) / (df['BB_Upper'] - df['BB_Lower'] + 1e-6)

# Volatility & Volume Indicators
df['Volatility_20'] = df['Return_1d'].rolling(20).std()
df['Vol_Ratio'] = df['Volume'] / (df['Volume'].rolling(20).mean() + 1e-6)
df['Deliverable_Pct'] = df['%Deliverble']

# Target Variable: 5-Day Forward Trend Direction (1 = Higher, 0 = Lower/Flat)
df['Target'] = (df['Close'].shift(-5) > df['Close']).astype(int)

feature_cols = [
    'Return_1d', 'Return_5d', 'MACD', 'MACD_Hist', 'BB_Pct', 
    'Volatility_20', 'Vol_Ratio', 'Deliverable_Pct'
]

df_clean = df.dropna().iloc[:-5].reset_index(drop=True)
X = df_clean[feature_cols]
y = df_clean['Target']

print(f"Clean Samples: {len(X)}")
print(f"Class Distribution: Up = {sum(y == 1)}, Down = {sum(y == 0)}")

## 3. Train-Test Split (80:20) & Feature Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training Set: {X_train.shape[0]} rows (80%)")
print(f"Testing Set:  {X_test.shape[0]} rows (20%)")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Model Training & Comparison Leaderboard

In [ ]:
models = {
    'Random Forest (Tuned)': RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_split=4, random_state=42),
    'XGBoost Classifier': XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, eval_metric='logloss', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, learning_rate=0.08, max_depth=5, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=300, max_depth=10, random_state=42),
    'Support Vector Machine (RBF)': SVC(kernel='rbf', C=2.0),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7)
}

results = []
trained = {}

for name, m in models.items():
    m.fit(X_train_scaled, y_train)
    trained[name] = m
    pred = m.predict(X_test_scaled)
    acc = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred, zero_division=0)
    rec = recall_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred, zero_division=0)
    results.append({
        'Model': name,
        'Accuracy (%)': round(acc * 100, 2),
        'Precision (%)': round(prec * 100, 2),
        'Recall (%)': round(rec * 100, 2),
        'F1-Score (%)': round(f1 * 100, 2)
    })

leaderboard = pd.DataFrame(results).sort_values('Accuracy (%)', ascending=False).reset_index(drop=True)
leaderboard

## 5. Champion Model & Feature Importances

In [ ]:
champion = trained['Random Forest (Tuned)']
importances = pd.Series(champion.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("=== Feature Importance Breakdown ===")
for feat, imp in importances.items():
    print(f"  • {feat:16s}: {imp * 100:.2f}%")